# CS-Jail-UR: background GPU run (generation only)

Runs headless with **Save Version → Save & Run All (Commit)**, so closing the
browser cannot interrupt it (limit 12 h; this run needs about 3.5–4.5 h).

Before saving:
1. Settings → Accelerator **GPU T4 x2**, Internet **On**.
2. Add-ons → Secrets → `HF_TOKEN` with **Attached** ticked. No OpenAI key is
   needed: nothing is judged in this run.
3. Set the flags in the first code cell.

When the version finishes, download `gpu_outputs.zip` from the version's
**Output** tab. Keep this notebook **private**: its outputs contain harmful
prompts and model responses.

In [ ]:
# Set these before "Save & Run All".
REGENERATE_EXP1 = True       # False if you already downloaded exp1_stageA.zip (raters may have it)
RUN_EXP2_MAIN = True         # 9,492 greedy generations: 3 models x 4 conditions x 791 families
RUN_EXP2_ROBUSTNESS = True   # 6,000 sampled generations: 200 families x CS/RU x 5 draws x 3 models

import os
from IPython import get_ipython


def sh(cmd):
    """Run a shell command with streamed output; stop the whole run if it fails."""
    print("$", cmd, flush=True)
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"step failed (exit {code}): {cmd}")


def package():
    """Zip everything produced so far into /kaggle/working/gpu_outputs.zip (overwrites)."""
    keep = [p for p in ("outputs/exp1/validation-kaggle-01", "outputs/exp1/benign-validation-kaggle-01",
                        "outputs/exp2", "outputs/checks") if os.path.exists(p)]
    sh("rm -f /kaggle/working/gpu_outputs.zip && zip -qr /kaggle/working/gpu_outputs.zip " + " ".join(keep))

In [ ]:
# Repo + Python 3.12 environment for the pinned stack (Kaggle's own Python is 3.13).
%cd /kaggle/working
if not os.path.isdir("NLP_Research"):
    sh("git clone --branch revision/final-692-dataset https://github.com/mshamoonbutt/NLP_Research.git")
%cd /kaggle/working/NLP_Research
sh("git pull --ff-only && git log --oneline -1")
sh("python -m pip install -q uv")
sh("python -m uv venv --allow-existing --python 3.12 /root/py312")
sh('python -m uv pip install --python /root/py312/bin/python -e ".[gpu,dev]"')
os.environ["PATH"] = "/root/py312/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = "/root/py312"
os.environ.pop("PYTHONPATH", None)
sh("which python && python --version")

In [ ]:
# Secret, GPU, frozen dataset, test suite. Any failure stops the run here.
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
sh("nvidia-smi -L")
sh("python scripts/exp0_finalize_data.py --restore")
sh("python -m pytest tests -q -p no:cacheprovider")

In [ ]:
# Exp 1 validation samples (same seeded families as before) + Urdu-script check.
if REGENERATE_EXP1:
    sh("python scripts/exp1_sample_for_annotation.py --role validation --models qwen25 phi3 llama32 "
       "--exclude-sample-dirs outputs/exp1/development-cpu-20260926 outputs/exp6/feasibility-cpu-20260927 "
       "--chunk-size 48 --out-dir outputs/exp1/validation-kaggle-01")
    sh("python scripts/benign_sample_for_annotation.py --role validation --models phi3 llama32 "
       "--out-dir outputs/exp1/benign-validation-kaggle-01")
    try:  # evidence only; must not block Exp 2
        sh("python scripts/urdu_sanity_check.py --backend vllm --models qwen25 phi3 llama32 "
           "--out outputs/checks/urdu_sanity_vllm.json")
    except RuntimeError as e:
        print("WARN (non-critical):", e)

In [ ]:
# Exp 2, generation only. Judged later from the cache once the judge PASSes (nothing regenerated).
if RUN_EXP2_MAIN:
    sh("python -m csjail.run_eval --out-dir outputs/exp2/main --skip-judge")
    package()  # the main sweep is safe in the zip even if a later step fails
if RUN_EXP2_ROBUSTNESS:
    try:  # bounded, so a stall cannot consume the session's 12 h
        sh("timeout 3h python scripts/exp2_robustness.py --skip-judge --greedy-results outputs/exp2/main")
    except RuntimeError as e:
        print("WARN: robustness incomplete (exit 124 = 3 h timeout):", e)

In [ ]:
# Package everything for one download from the Output tab.
package()
sh("ls -lh /kaggle/working/gpu_outputs.zip")
print("DONE: download gpu_outputs.zip from this version's Output tab.")